In [292]:
import sys
from pathlib import Path
import polars as pl
from pymongo import MongoClient

## Configuration

In [ ]:
# 🔧 remonte d'un niveau depuis analysis/notebooks/ vers analysis/
root_dir = Path().resolve().parent
sys.path.append(str(root_dir))

from utils import tools

In [ ]:
MONGO_CONNECTION_STR = "mongodb://localhost:27017/"
DB_NAME = "short_term_rentals"
PARIS_COLLECTION = "paris_listings"

## Connection MongoDB

In [295]:
client = MongoClient(MONGO_CONNECTION_STR)
db = client[DB_NAME]
paris_collection = db[PARIS_COLLECTION]

### Test de connection

In [296]:
try:
    one_document = paris_collection.find_one()
    if one_document:
        print("✅ Connexion à MongoDB réussie !")
        print(f"   Un document a été récupéré (ID: {one_document['_id']})")
    else:
        print("🟡 Connexion réussie, mais la collection est vide.")
except Exception as e:
    print(f"❌ Erreur de connexion à MongoDB : {e}")

✅ Connexion à MongoDB réussie !
   Un document a été récupéré (ID: 6912df0f2f1667e183acb853)


## Extraction des données et chargement dans Polars

In [ ]:
cursor = paris_collection.find({}, { "_id": 0 })
# infer_schema_length=None : scan all documents before inferring types
# required because some fields (e.g. amenities) have empty arrays [] in early documents
df_paris = pl.DataFrame(list(cursor), infer_schema_length=None)
print(f"✅ DataFrame chargé avec succès. Forme : {df_paris.shape}")

Les requêtes à réaliser sont les suivantes :

- Calculer le taux de réservation moyen par mois par type de logement
- Calculer la médiane des nombre d’avis pour tous les logements
- Calculer la médiane des nombre d’avis par catégorie d’hôte
- Calculer la densité de logements par quartier de Paris
- Identifier les quartiers avec le plus fort taux de réservation par mois

## Analyse complexes avec Polars

### 1. Taux de réservation moyen par type de logement

In [298]:
# Observons déjà la fiabilité de champs qui nous concernent soit has_availability et availability_30
nb_listings = df_paris.height

# Le cas d'incohérence : Annonces marquées "inactives" (has_availability: False) mais qui ont quand même des jours disponibles dans leur calendrier.
incoherent_listings_count = df_paris.filter(
    (pl.any_horizontal(pl.col("^availability_.*$") > 0)) & 
    ~(pl.col("has_availability"))
).height

listings_with_coherent_status_count = nb_listings - incoherent_listings_count
reliability_rate = (listings_with_coherent_status_count / nb_listings) * 100

# Affichage des résultats pour 'has_availability'
tools.print_md(
    "**Analyse du champs `has_availability`**  \n",         
    f"*Taux des valeurs manquantes:* **{
        round(df_paris["has_availability"].null_count() / nb_listings, 4)*100
    } %**",
    f"*Nombre d'annonces avec un statut incohérent :* **{incoherent_listings_count}**",
    f"*Taux de fiabilité:* **{reliability_rate:.2f} %**",
)

# Affichage des résultats pour 'availability_30'
tools.print_md(
    "**Analyse du champs `availability_30`**  \n",         
    f"*Taux des valeurs manquantes:* **{
        round(df_paris["availability_30"].null_count() / nb_listings, 4)*100
    } %**",
)

**Analyse du champs `has_availability`**  


*Taux des valeurs manquantes:* **5.09 %**

*Nombre d'annonces avec un statut incohérent :* **694**

*Taux de fiabilité:* **99.28 %**

**Analyse du champs `availability_30`**  


*Taux des valeurs manquantes:* **0.0 %**

In [299]:
df_booking_rate = (
                df_paris
                    .filter(
                        (pl.col("availability_30").is_not_null()) 
                        & (pl.col("has_availability") | pl.col("has_availability").is_null())
                    )
                    .with_columns([
                    ((30 - pl.col("availability_30")) / 30 * 100).alias("booking_rate_30d_pct")
                ])
)
df_booking_rate.height

95057

In [300]:
(
    df_booking_rate
        .group_by("room_type")
        .agg([
            pl.col("booking_rate_30d_pct").mean().alias("avg_booking_rate"),
            pl.len().alias("nb_listings")
            ]).sort("avg_booking_rate", descending=True)
)

room_type,avg_booking_rate,nb_listings
str,f64,u32
"""Entire home/apt""",71.293528,85031
"""Private room""",70.147765,8888
"""Shared room""",60.816667,400
"""Hotel room""",51.237579,738


### 2. Médiane du nombre d'avis pour tous les logements

In [301]:
# Analyse de valeurs manquantes pour 'number_of_reviews'
tools.print_md(
    "**Analyse du champs `number_of_reviews`**  \n",         
    f"*Taux des valeurs manquantes:* **{
        round(df_paris["number_of_reviews"].null_count() / nb_listings, 4)*100
    } %**",
)

**Analyse du champs `number_of_reviews`**  


*Taux des valeurs manquantes:* **0.0 %**

In [302]:
median_reviews_all = df_paris.select(
                        pl.col("number_of_reviews")
                            .median()
                            .alias("median_reviews")
)
median_reviews_all

median_reviews
f64
3.0


### 3. Médiane du nombre d'avis par catégorie d'hôte
**Conclusion** :
- Le statut de "Super-hôte" est extrêmement corrélé à un nombre d'avis beaucoup plus élevé. 
- Un super-hôte type a significativement plus d'avis (et donc probablement plus d'activité et/ou une meilleure satisfaction client) qu'un hôte standard.

In [303]:
# Analyse de valeurs manquantes pour 'host_is_superhost'
tools.print_md(
    "**Analyse du champs `host_is_superhost`**  \n",         
    f"*Taux des valeurs manquantes:* **{
        round(df_paris["host_is_superhost"].null_count() / nb_listings, 4)*100
    } %**",
)

**Analyse du champs `host_is_superhost`**  


*Taux des valeurs manquantes:* **0.08 %**

In [304]:
# Pour chaque groupe (true/false), on calcule la médiane de 'number_of_reviews'
mapping = {
    True: "super host",
    False: "regular host",
    None: "unknown"
}

median_reviews_by_superhost = (
            df_paris
                .group_by("host_is_superhost")
                .agg(pl.col("number_of_reviews").median().alias("median_reviews"))
                .with_columns(
                    pl.col("host_is_superhost")
                    .replace_strict(mapping)
                    .alias("host_type")
                )
                .drop("host_is_superhost") 
                .select(["host_type", "median_reviews"]) 
                .sort("median_reviews")
)                  
median_reviews_by_superhost

host_type,median_reviews
str,f64
"""regular host""",2.0
"""unknown""",12.5
"""super host""",24.0


### 4. Densité de logements par quartier

In [305]:
# Affichage des résultats pour 'neighbourhood'
tools.print_md(
    "**Analyse du champs `neighbourhood`**  \n",         
    f"*Taux des valeurs manquantes:* **{
        round(df_paris["neighbourhood"].null_count() / nb_listings, 4)*100
    } %**",
)

# Affichage des résultats pour 'neighbourhood_cleansed'
tools.print_md(
    "**Analyse du champs `neighbourhood_cleansed`**  \n",         
    f"*Taux des valeurs manquantes:* **{
        round(df_paris["neighbourhood_cleansed"].null_count() / nb_listings, 4)*100
    } %**",
)

**Analyse du champs `neighbourhood`**  


*Taux des valeurs manquantes:* **48.52 %**

**Analyse du champs `neighbourhood_cleansed`**  


*Taux des valeurs manquantes:* **0.0 %**

In [312]:
# On regroupe par 'neighbourhood_cleansed' et on compte le nombre d'annonces (lignes)
listings_density_by_neighborhood = (
                df_paris
                    .filter(
                        (pl.col("has_availability") | pl.col("has_availability").is_null())
                    )
                    .group_by("neighbourhood_cleansed")
                    .agg(pl.len().alias("number_of_listings"))
                    .sort("number_of_listings", descending=True)
)
listings_density_by_neighborhood.head(10)

neighbourhood_cleansed,number_of_listings
str,u32
"""Buttes-Montmartre""",10478
"""Popincourt""",8354
"""Vaugirard""",7710
"""Batignolles-Monceau""",6792
"""Entrepôt""",6498
"""Passy""",6344
"""Buttes-Chaumont""",5412
"""Ménilmontant""",5230
"""Opéra""",4545


### 5. Quartiers avec le plus fort taux de réservation

In [313]:
df_booking_rate.group_by("neighbourhood_cleansed").agg([
    pl.col("booking_rate_30d_pct").mean().round(2).alias("avg_booking_rate"),
    pl.len().alias("nb_listings")
]).sort("avg_booking_rate", descending=True)

neighbourhood_cleansed,avg_booking_rate,nb_listings
str,f64,u32
"""Ménilmontant""",75.39,5230
"""Popincourt""",74.79,8354
"""Entrepôt""",74.72,6498
"""Buttes-Chaumont""",74.17,5412
"""Buttes-Montmartre""",73.1,10478
…,…,…
"""Palais-Bourbon""",68.97,2722
"""Bourse""",68.63,3031
"""Luxembourg""",66.48,2689


In [309]:
(
    df_paris
        .filter(
            (pl.col("availability_30").is_not_null()) & pl.col("has_availability")
        )
        .with_columns([        
            (((30 - pl.col("availability_30")) / 30) * 100)
            .alias("booking_rate_30d_pct")
        ])
        .group_by("neighbourhood_cleansed")
        .agg([
            pl.col("booking_rate_30d_pct").mean().alias("avg_booking_rate"),
            pl.len().alias("nb_listings") # On garde le nombre d'annonces pour le contexte
        ])
        .sort("avg_booking_rate", descending=True)
)

neighbourhood_cleansed,avg_booking_rate,nb_listings
str,f64,u32
"""Ménilmontant""",74.845967,4934
"""Popincourt""",74.428212,7940
"""Entrepôt""",74.311461,6233
"""Buttes-Chaumont""",73.797559,5134
"""Buttes-Montmartre""",72.681687,9898
…,…,…
"""Palais-Bourbon""",68.090452,2587
"""Bourse""",67.804795,2920
"""Luxembourg""",65.988395,2585
